# PAMWeave：从数据校验到候选清单

先完成主 README 的依赖、固定权重和上游适配层配置。这里调用与命令行完全相同的入口，不另写评分公式。首次运行会进行真实模型推理；已有输出时复用缓存。可将 `FRESH=True` 强制重跑。

默认队列是21个可再分发系统，生成20行候选；完整22系统参考结果另有25行。分数是模型兼容度或侧翼支持，不是实验活性。

In [1]:
from pathlib import Path
import csv, json, os, subprocess, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.sh").is_file() and (p / "src/pamdict").is_dir())
MANIFEST = "benchmarks/multi_evidence_v2/manifest_redistributable21.json"
manifest = json.loads((ROOT / MANIFEST).read_text())
print("可再分发系统数：", len(manifest["systems"]))
print("权重与数据获取方式：models/MODEL_CARD.md、data/README.md")

可再分发系统数： 21
权重与数据获取方式：models/MODEL_CARD.md、data/README.md


## 1. 校验输入
检查清单契约与每个输入的 SHA-256，不加载模型。

In [2]:
check = subprocess.run([sys.executable, "scripts/validate_benchmark_manifest.py", "--manifest", MANIFEST, "--check-files"], cwd=ROOT, check=True, capture_output=True, text=True)
audit = json.loads(check.stdout)
assert audit["valid"]
print("输入校验：", audit["valid"])
print("系统数：", audit["system_count"])
print("全部清单登记文件已完成校验。")

输入校验： True
系统数： 21
全部清单登记文件已完成校验。


## 2. 执行主流程
蛋白模型推理、侧翼分析、证据对照、评价和候选导出由主入口执行。普通日志写入 `logs/runs/`。设置 `PAMWEAVE_NOTEBOOK_OUTDIR` 可选用已有分析目录；这是目录配置，不改变候选评分。

In [3]:
FRESH = False
outdir = os.environ.get("PAMWEAVE_NOTEBOOK_OUTDIR", "data/parsed/bench_repro21")
command = [sys.executable, "scripts/run_submission.py", "--device", "cpu", "--outdir", outdir]
if FRESH:
    command.append("--fresh")
run = subprocess.run(command, cwd=ROOT, check=True, capture_output=True, text=True)
print("\n".join(run.stdout.splitlines()[-4:]))
log_path = next(line.split(": ", 1)[1] for line in run.stdout.splitlines() if line.startswith("Execution record: "))
record = json.loads((ROOT / log_path / "run.json").read_text())
assert record["exit_code"] == 0
print("Bootstrap种子：", record["bootstrap_seed"])
print("模型推理缓存数（运行开始时）：", len(record["systems_with_existing_protein_outputs_at_start"]))
print("完整执行耗时（秒）：", record["seconds"])

wrote 20 rows -> <PACKAGE_ROOT>/results/results_repro21.csv
Candidate list: results/results_repro21.csv
Execution record: logs/runs/20261003T164736_393105Z
Exit code: 0
Bootstrap种子： 17
模型推理缓存数（运行开始时）： 21
完整执行耗时（秒）： 22.505


## 3. 检查生成的候选清单
保留蛋白侧与侧翼侧的原始指标。缺失值留空，避免解释为零支持。

In [4]:
results = ROOT / next(path for path in record["outputs"] if path.endswith(".csv"))
with results.open(encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle))
expected_rows = 25 if record["manifest"].endswith("manifest_final22.json") else 20
assert len(rows) == expected_rows
assert all(row["candidate_id"] and row["model_version"] and row["code_version"] for row in rows)
assert all(row["support_target_count"] == "" for row in rows if row["evidence_source"] == "published_aggregate_flank")
print("候选行数：", len(rows))
print("展示系统数：", len({row["cas9_id"] for row in rows}))
for row in rows[:5]:
    print(row["cas9_id"], row["candidate_pam"], row["protein_rank_in_length"], row["evidence_status"])
print("参考完整22系统的25行结果：results/results.csv")

候选行数： 20
展示系统数： 4
WP_038431314.1 AGG 2 protein_support_spacer_neutral
WP_038431314.1 CGG 3 concordant_support_limited_independence
WP_038431314.1 GGG 4 concordant_support_limited_independence
WP_038431314.1 TGA 5 protein_neutral_spacer_conflict
WP_038431314.1 TGG 1 concordant_support_limited_independence
参考完整22系统的25行结果：results/results.csv


## 解释边界
本 Notebook 执行复现与结果核查；不构成独立泛化验证或湿实验验证。严格独立配对系统不足，当前结果不支持稳定融合提升。实际执行记录保留在上一步输出的日志目录。